# Что видит машинаПеред вами 20 секунд съёмки с автомобиля: пять камер, дорога, люди, техника.Модель никто не обучал под эту задачу — она уже умеет находить объекты **по описанию словами**.Ваша задача: подобрать запрос и настройки так, чтобы система насчитала столько объектов,сколько их было на самом деле.**Как работать:** запускайте ячейки сверху вниз кнопкой ▶ или `Shift+Enter`.Менять нужно только ячейку с надписью **РУЧКИ** — всё остальное просто выполняйте.

## 1. ПодготовкаОдин раз за сессию. Модель большая, загрузка занимает несколько секунд.

In [ ]:
import matplotlib.pyplot as pltfrom stand import data, run, vizplt.rcParams["figure.dpi"] = 110clip_all = data.segment_summary()clip_all

## 2. Что в записиПять камер, у каждой 199 кадров. В таблице выше — сколько размеченных объектоввидит каждая камера. Эта разметка сделана людьми и будет нашим эталоном:с ней мы сравним то, что найдёт модель.

## 3. Берём отрезокСорок кадров передней камеры — четыре секунды езды.

In [ ]:
clip = data.load_clip(n_frames=40, camera=1)print(f"кадров: {len(clip)}, размер: {clip.size[0]}x{clip.size[1]}")print("\nСколько объектов реально прошло за отрезок:")print(clip.truth_counts().to_string())viz.show(clip.frames[0], "Первый кадр отрезка");

## 4. РУЧКИЭто единственная ячейка, которую вы меняете.| Ручка | Что делает ||---|---|| `ЗАПРОС` | **главная.** Слово или фраза, которую ищет модель || `ПОРОГ` | ниже этой уверенности объект отбрасывается. Выше порог — меньше находок, но чище || `МИН_ДЛИНА` | трек короче стольких кадров считается случайным и не засчитывается || `ЦЕЛЬ` | с каким классом эталона сравниваем: `VEHICLE`, `PEDESTRIAN` или `CYCLIST` |

In [ ]:
ЗАПРОС    = "vehicle"ПОРОГ     = 0.5МИН_ДЛИНА = 1ЦЕЛЬ      = "VEHICLE"

## 5. ЗапускСорок кадров считаются секунд двадцать.

In [ ]:
result = run.find(clip, ЗАПРОС, conf=ПОРОГ, min_track_len=МИН_ДЛИНА)итог = run.score(result, clip, target=ЦЕЛЬ)for k, v in итог.items():    print(f"{k:>18}: {v}")

## 6. Что она увиделаКаждый объект — свой цвет. Цвет закреплён за номером: если на разных кадрахобъект одного цвета, значит система считает, что это один и тот же объект.

In [ ]:
кадр = 0det = result.detections[result.detections["кадр"] == кадр]viz.show(viz.draw(clip.frames[кадр], det, result.masks.get(кадр)),         f"Запрос «{ЗАПРОС}», порог {ПОРОГ} — найдено {len(det)}");

## 7. Не теряет ли она объектыЧетыре кадра подряд. Смотрите на цвета: если объект меняет цвет от кадра к кадру,система потеряла его и завела заново — а значит посчитает дважды.

In [ ]:
viz.grid(clip, result, frames=(0, 12, 24, 36));

## 8. Сравнение с эталоном по кадрам

In [ ]:
viz.counts_plot(result, clip, target=ЦЕЛЬ);

## 9. Порог — это решение, а не настройкаОдин и тот же прогон, разные отсечки. Красная линия — правда.Подняли порог: отсеяли мусор, но потеряли настоящие объекты.Опустили: нашли всё, но вместе с выдумками. Золотой середины не существует —есть выбор, что дороже. На производстве этот выбор делает не разработчик.

In [ ]:
кривая = run.sweep_conf(clip, ЗАПРОС, target=ЦЕЛЬ)viz.sweep_plot(кривая, target=ЦЕЛЬ)кривая

## 10. Одно слово меняет ответМодель не «знает объекты». Она отвечает на **заданный вопрос**.Спросите иначе — получите другое число.

In [ ]:
таблица = run.compare_prompts(clip, ["car", "vehicle", "truck", "bus"],                              conf=ПОРОГ, target="VEHICLE")viz.prompts_plot(таблица)таблица

## 11. А теперь — сколько до них метровРядом с камерой стоит лазерный дальномер. Он не видит смысла, зато честно меряетрасстояние до каждой точки. Сейчас мы наложим его измерения на фотографию.

In [ ]:
from stand import lidarlidar.overlay(clip, frame=0);

## 12. Слово → контур → метрыМодель нашла объекты по слову. Дальномер сказал, как далеко каждый.Связка идёт по контуру: берём точки, попавшие внутрь маски.**Модель отвечает «что и где». Прибор отвечает «сколько». Ни один из них не может оба.**

In [ ]:
lidar.distance_to_objects(result, clip, frame=0)

## 13. Кто приближаетсяРасстояние на двух кадрах — и получается скорость.

In [ ]:
lidar.speed_of_objects(result, clip, frames=(0, 10))

## 14. Вид сверхуТа же секунда, но с высоты. Красный треугольник — наша машина.Именно так сцену видит беспилотник.

In [ ]:
lidar.bev(clip, frame=0);

## 15. Можно и вообще без нейросетейУбираем дорогу по высоте, оставшиеся точки разбиваем на связные скопления.Ни одной обученной модели — чистая геометрия, доли секунды.Сравните с тем, что нашла модель по слову: ошибки у них **разные**.Камера путается в тенях и бликах, дальномер не отличает столб от человека.

In [ ]:
from stand import scenefig, объекты = scene.show_geometry(clip, frame=0)объекты.head(10)

## 16. Глубина по одной фотографииДальномер есть не везде. Но расстояние можно **предсказать** по обычному снимку —и проверить по дальномеру, насколько это правда.

In [ ]:
from stand import depthfig, качество = depth.compare(clip, frame=0)качество

Насколько предсказание совпало с измерением по каждому найденному объекту:

In [ ]:
depth.distance_table(result, clip, frame=0)

## 17. Как эту сцену разметил человекДвадцать девять классов, размеченных вручную: дорога, тротуар, здания, растительность.Это тот самый дорогой ручной труд, о котором идёт речь в докладе, — и вот сколькоего нужно, чтобы получить один такой кадр.

In [ ]:
кадры_с_разметкой = scene.labeled_frames(clip)scene.show_panoptic(clip, кадры_с_разметкой[0])scene.class_areas(clip, кадры_с_разметкой[0]).head(10)

## 18. Сдать результатВернитесь к ячейке **РУЧКИ**, поменяйте запрос или порог, прогоните заново —и когда число сойдётся с эталоном, отправляйте.

In [ ]:
run.save_submission(result, clip, target=ЦЕЛЬ, path="submission.csv")